<a href="https://colab.research.google.com/github/Leashaniya/Research-Project/blob/leasha/05_RAG_QUESTION_GENERATION_(PP1%2C_MAIN_QUESTIONS_ONLY).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
# ==========================================================
# NOTEBOOK 5 (v2) — RAG MODEL PAPER GENERATION (NO OPENAI)
# Fixes:
#  - Cleans slide context (removes slide markers / junk)
#  - Forces output format (MCQ / Structured)
#  - Rejects garbage outputs + regenerates automatically
#  - Removes [SOURCE: ...] leaks from final question
# ==========================================================

from google.colab import drive
drive.mount("/content/drive")

!pip -q install sentence-transformers faiss-cpu transformers accelerate bitsandbytes

import json, re, random
from pathlib import Path
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# -------------------------------
# Paths
# -------------------------------
RP_ROOT   = Path("/content/drive/MyDrive/RP")

TEMPLATES_PATH = RP_ROOT / "template_questions.json"
BLUEPRINT_PATH = RP_ROOT / "exam_blueprint_template.json"

SLIDES_CHUNKS_PATH = RP_ROOT / "lecture_slides_extraction" / "slides_chunks.jsonl"
SLIDES_META_PATH   = RP_ROOT / "slides_embeddings" / "slides_metadata.jsonl"
SLIDES_FAISS_PATH  = RP_ROOT / "slides_embeddings" / "slides_faiss_index_flatip.index"

OUT_JSON = RP_ROOT / "model_exam_paper.json"
OUT_TXT  = RP_ROOT / "model_exam_paper.txt"

for p in [TEMPLATES_PATH, BLUEPRINT_PATH, SLIDES_CHUNKS_PATH, SLIDES_META_PATH, SLIDES_FAISS_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required file: {p}")

print("✅ All required files found.")

templates = json.loads(TEMPLATES_PATH.read_text(encoding="utf-8"))
exam_blueprint = json.loads(BLUEPRINT_PATH.read_text(encoding="utf-8"))

slide_chunks = [json.loads(l) for l in SLIDES_CHUNKS_PATH.read_text(encoding="utf-8").splitlines() if l.strip()]
slide_meta   = [json.loads(l) for l in SLIDES_META_PATH.read_text(encoding="utf-8").splitlines() if l.strip()]

index = faiss.read_index(str(SLIDES_FAISS_PATH))
print("✅ Loaded FAISS index. vectors:", index.ntotal)

# -------------------------------
# Models
# -------------------------------
retriever_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Use base if you're on CPU (large is slow and more likely to degrade)
# If you enable GPU runtime, you can switch back to flan-t5-large.
FLAN_MODEL_NAME = "google/flan-t5-base"
# FLAN_MODEL_NAME = "google/flan-t5-large"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
print("Loading generator:", FLAN_MODEL_NAME)

tokenizer = AutoTokenizer.from_pretrained(FLAN_MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(FLAN_MODEL_NAME).to(device)
model.eval()
print("✅ Generator loaded.")

# -------------------------------
# Utility: normalize for cosine search
# -------------------------------
def normalize_l2(x: np.ndarray) -> np.ndarray:
    x = x.astype("float32")
    norm = np.linalg.norm(x, axis=1, keepdims=True) + 1e-12
    return x / norm

# -------------------------------
# Context cleaning (CRITICAL)
# -------------------------------
BAD_CONTEXT_PATTERNS = [
    r"\[SOURCE:.*?\]",
    r"---\s*SLIDE\s*\d+\s*---",
    r"---\s*PAGE\s*\d+\s*---",
]

def clean_context(text: str) -> str:
    t = text or ""
    for pat in BAD_CONTEXT_PATTERNS:
        t = re.sub(pat, " ", t, flags=re.I|re.S)

    # remove repeated punctuation/quotes/dashes
    t = re.sub(r'["\']{2,}', " ", t)
    t = re.sub(r"[-_]{3,}", " ", t)
    t = re.sub(r"\s+", " ", t).strip()

    # remove very long “table-like” lines (often garbage)
    # keep only moderately sized chunks
    if len(t) > 1800:
        t = t[:1800]

    return t.strip()

# -------------------------------
# Retrieval
# -------------------------------
def retrieve_slide_context(query_text: str, top_k: int = 8, max_chars: int = 1600):
    q_emb = retriever_model.encode([query_text]).astype("float32")
    q_emb = normalize_l2(q_emb)
    D, I = index.search(q_emb, top_k)

    hits = []
    parts = []
    for idx, score in zip(I[0].tolist(), D[0].tolist()):
        if idx < 0:
            continue
        chunk = slide_chunks[idx]
        meta  = slide_meta[idx] if idx < len(slide_meta) else {}
        raw = (chunk.get("text") or "").strip()
        cleaned = clean_context(raw)

        if not cleaned or len(cleaned.split()) < 20:
            continue

        hits.append({
            "score": float(score),
            "chunk_id": meta.get("chunk_id", chunk.get("chunk_id")),
            "pdf_stem": meta.get("pdf_stem", chunk.get("pdf_stem")),
            "slide_no": meta.get("slide_no", chunk.get("slide_no")),
        })
        parts.append(cleaned)

        if sum(len(p) for p in parts) > max_chars:
            break

    context = "\n".join(parts).strip()
    if len(context) > max_chars:
        context = context[:max_chars].rsplit(" ", 1)[0] + "..."

    return context, hits

# -------------------------------
# Output validation
# -------------------------------
def looks_like_garbage(text: str) -> bool:
    t = (text or "").strip()
    if len(t) < 30:
        return True
    if "[SOURCE" in t or "--- SLIDE" in t or "--- PAGE" in t:
        return True
    # too many quotes/dashes = garbage
    if len(re.findall(r'["\']', t)) > 20:
        return True
    if len(re.findall(r"-", t)) > 40:
        return True
    # repeated token patterns like "" - "" - ""
    if re.search(r'""\s*-\s*""', t):
        return True
    # very low alphabetic ratio
    alpha = sum(c.isalpha() for c in t)
    if alpha / max(1, len(t)) < 0.35:
        return True
    return False

def validate_mcq(text: str) -> bool:
    t = (text or "").strip()
    # must have A) B) C) D) (or A. etc)
    ok = all(re.search(rf"\b{opt}\s*[\)\.\:]", t) for opt in ["A", "B", "C", "D"])
    return ok and (not looks_like_garbage(t))

def validate_structured(text: str) -> bool:
    t = (text or "").strip()
    # must have at least (a) and (b)
    has_parts = bool(re.search(r"\(a\)", t, flags=re.I) and re.search(r"\(b\)", t, flags=re.I))
    return has_parts and (not looks_like_garbage(t))

def enforce_clean_final(text: str) -> str:
    t = (text or "").strip()
    t = re.sub(r"\[SOURCE:.*?\]", " ", t, flags=re.I|re.S)
    t = re.sub(r"---\s*SLIDE\s*\d+\s*---", " ", t, flags=re.I)
    t = re.sub(r"\s+", " ", t).strip()
    return t

# -------------------------------
# Prompt templates by marks
# -------------------------------
def prompt_for_mcq(template_text: str, context: str, pattern_label: str):
    return f"""
You are a university lecturer creating ONE MCQ question for Database Management Systems.

Use ONLY the CONTEXT facts (do not invent).
The MCQ must be clear and self-contained.
Output MUST follow exactly this format:

Question: <one sentence question>
A) <option>
B) <option>
C) <option>
D) <option>

No answers. No explanations.

Pattern: {pattern_label}

TEMPLATE STYLE (reference only):
{template_text}

CONTEXT:
{context}
""".strip()

def prompt_for_structured(template_text: str, context: str, pattern_label: str, marks: int):
    return f"""
You are a university lecturer creating ONE structured exam question for Database Management Systems worth about {marks} marks.

Use ONLY the CONTEXT facts (do not invent).
Output MUST include subparts (a) and (b) (and optionally (c)).
Output ONLY the question (no answers).

Pattern: {pattern_label}

Format required:
<Main question sentence>
(a) ...
(b) ...
(c) ...

TEMPLATE STYLE (reference only):
{template_text}

CONTEXT:
{context}
""".strip()

# -------------------------------
# Generation
# -------------------------------
def generate_text(prompt: str, max_new_tokens=240):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            num_beams=1,
            repetition_penalty=1.15,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True).strip()

def generate_question(template_obj: dict, marks: int, max_tries: int = 6):
    template_text = (template_obj.get("full_text") or "").strip()
    pattern_label = template_obj.get("pattern_label", "GENERAL")

    context, hits = retrieve_slide_context(template_text, top_k=10, max_chars=1600)
    if not context:
        # fallback context
        context = clean_context(template_text)

    # Decide format:
    # For 20 or less -> MCQ (because your Q1 looked like MCQ)
    # For >20 -> structured
    want_mcq = marks <= 20

    for attempt in range(1, max_tries + 1):
        if want_mcq:
            prompt = prompt_for_mcq(template_text, context, pattern_label)
            gen = generate_text(prompt, max_new_tokens=220)
            gen = enforce_clean_final(gen)
            if validate_mcq(gen):
                return gen, hits, "MCQ"
        else:
            prompt = prompt_for_structured(template_text, context, pattern_label, marks)
            gen = generate_text(prompt, max_new_tokens=280)
            gen = enforce_clean_final(gen)
            if validate_structured(gen):
                return gen, hits, "STRUCTURED"

    # If all fails, return a safe structured shell (better than garbage)
    safe = f"{template_text}\n(a) Explain the relevant concept.\n(b) Apply it to a given scenario."
    return enforce_clean_final(safe), hits, "FALLBACK"

# -------------------------------
# Assemble paper from blueprint slots
# -------------------------------
slots = exam_blueprint.get("question_slots", [])
if not slots:
    raise ValueError("No question_slots found in exam blueprint.")

templates_by_marks = {}
for t in templates:
    m = t.get("marks")
    if m is None:
        continue
    templates_by_marks.setdefault(int(m), []).append(t)

def pick_template_for_mark(mark: int):
    pool = templates_by_marks.get(int(mark), [])
    if pool:
        return random.choice(pool)
    return random.choice(templates)

random.seed(42)

generated_questions = []
print("\n=== Generating questions for each slot ===")
for s in slots:
    slot_id = s["slot_id"]
    target_marks = int(s["target_marks"])
    print(f"\n{slot_id} | target_marks={target_marks}")

    t = pick_template_for_mark(target_marks)
    qtext, hits, fmt = generate_question(t, marks=target_marks, max_tries=6)

    generated_questions.append({
        "slot_id": slot_id,
        "position": int(s.get("position", 0)),
        "target_marks": target_marks,
        "format": fmt,
        "pattern_label": t.get("pattern_label"),
        "template_source": {"pdf_stem": t.get("pdf_stem"), "question_id": t.get("question_id")},
        "question_text": qtext,
        "retrieved_sources": hits[:5],
    })

total_marks = sum(q["target_marks"] for q in generated_questions)
expected_total = int(exam_blueprint.get("canonical_total_marks", 100))
print("\n=== Summary ===")
print("Generated questions:", len(generated_questions))
print("Total marks:", total_marks, "| expected:", expected_total)

# -------------------------------
# Save outputs
# -------------------------------
paper_json = {
    "component": "generated_model_exam_paper_rag_v2",
    "generator_model": FLAN_MODEL_NAME,
    "device": device,
    "canonical_total_marks_expected": expected_total,
    "canonical_total_marks_generated": total_marks,
    "questions": sorted(generated_questions, key=lambda x: x["position"])
}
OUT_JSON.write_text(json.dumps(paper_json, indent=2, ensure_ascii=False), encoding="utf-8")

lines = []
lines.append("MODEL EXAM PAPER (RAG-generated v2)")
lines.append(f"Generator: {FLAN_MODEL_NAME} | Device: {device}")
lines.append(f"Total Marks: {total_marks} / {expected_total}")
lines.append("=" * 70)

for q in sorted(generated_questions, key=lambda x: x["position"]):
    lines.append(f"\n{q['slot_id']} ({q['target_marks']} marks) — {q.get('pattern_label','')} — {q.get('format','')}")
    lines.append(q["question_text"].strip())

OUT_TXT.write_text("\n".join(lines).strip(), encoding="utf-8")

print("\n✅ Saved:")
print(" -", OUT_JSON)
print(" -", OUT_TXT)

print("\n--- Preview ---")
print("\n".join(lines[:30]))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ All required files found.
✅ Loaded FAISS index. vectors: 55
Device: cpu
Loading generator: google/flan-t5-base


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

✅ Generator loaded.

=== Generating questions for each slot ===

Q1 | target_marks=20

Q2 | target_marks=17

Q3 | target_marks=23

Q4 | target_marks=40

=== Summary ===
Generated questions: 4
Total marks: 100 | expected: 100

✅ Saved:
 - /content/drive/MyDrive/RP/model_exam_paper.json
 - /content/drive/MyDrive/RP/model_exam_paper.txt

--- Preview ---
MODEL EXAM PAPER (RAG-generated v2)
Generator: google/flan-t5-base | Device: cpu
Total Marks: 100 / 100

Q1 (20 marks) — ER_EER_MODELING — MCQ
A) option> B) option> C) option> D) option> A

Q2 (17 marks) — SQL_DDL_DML — FALLBACK
Question 4 (40 Marks) Consider the following relations in a database created for an online store Customers (cid: char (4), name: varchar (50), phone: char(10), country: varchar(20)) Employees (eid: char (4), ename: varchar (50), phone: char (4), hiredate: date) Orders (oid: int, eid: char (4), cid: char (4), orderDate: date, requiredDate: date, shippedDate: date) OrderDetails (oid: int, productld: char (4), quantit